In [36]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import AzureOpenAI
from embeddings import UserDocumentCollection
import os, json, re
import pandas as pd

import numpy as np
from pypdf import PdfReader
import win32com.client as win32
from time import sleep

In [37]:
load_dotenv()

True

In [38]:
raw_path = os.environ.get("RAW_PATH")
folder_path = os.environ.get("FOLDER_PATH")
data_path= Path(folder_path) / "PC"

In [39]:
files = [p for p in Path(raw_path).iterdir() if p.is_file()]
print("Number of files:", len(files))

Number of files: 92


In [40]:
from pathlib import Path
import re

raw_path = Path(raw_path)  # or Path(r"...full\path\here")

prefixes = set()

for p in raw_path.iterdir():
    if p.is_file() and p.suffix.lower() == ".pdf":
        prefix = re.split(r"[-_]", p.stem, maxsplit=1)[0]
        prefixes.add(prefix)

print("Number of unique PDF numbers:", len(prefixes))


Number of unique PDF numbers: 68


In [41]:
from pypdf import PdfReader, PdfWriter
import shutil
import re
from pathlib import Path

SKIP_NAME = "F33073064-Dutch_vision_on_biotechnology__ENGLISH.pdf"

def merge_pdfs(input_paths, output_path):
    writer = PdfWriter()
    for pdf_path in input_paths:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            writer.add_page(page)
    with open(output_path, "wb") as f:
        writer.write(f)

groups = {}

# 1) First, handle PDFs: group by prefix (before first "-" or "_")
for p in Path(raw_path).iterdir():
    if p.is_file():
        # ⬇️ skip this specific file
        if p.name == SKIP_NAME:
            continue

        if p.suffix.lower() == ".pdf":
            prefix = re.split(r"[-_]", p.stem, maxsplit=1)[0]
            groups.setdefault(prefix, []).append(p)
  

# 3) Now create exactly ONE PDF per prefix in output folder
for prefix, files in groups.items():
    files = sorted(files)
    out_pdf = data_path / f"{prefix}.pdf"

    if len(files) == 1:
        shutil.copy2(files[0], out_pdf)
        print(f"Copied single PDF for prefix '{prefix}' -> {out_pdf}")
    else:
        merge_pdfs(files, out_pdf)
        print(f"Merged {len(files)} PDFs for prefix '{prefix}' -> {out_pdf}")


Merged 2 PDFs for prefix 'F33073064' -> C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\data\PC\F33073064.pdf
Copied single PDF for prefix 'F33083732' -> C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\data\PC\F33083732.pdf
Copied single PDF for prefix 'F33094597' -> C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\data\PC\F33094597.pdf
Copied single PDF for prefix 'F33096176' -> C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\data\PC\F33096176.pdf
Copied single PDF for prefix 'F33097445' -> C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\data\PC\F33097445.pdf
Copied single PDF for prefix 'F33104014' -> C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\data\PC\F33104014.pdf
Copied single PDF for prefix 'F33105185' -> C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\data\PC\F33105185.pdf
Copied single PDF for prefix 'F33113117' -> C:\Users\YağmurAslan\OneDrive

In [20]:
container_name="brussels-docs"
colect = UserDocumentCollection(container_name)


for p in sorted([p for p in Path(data_path).iterdir() if p.is_file()]):
    file_path = str(p)
    with open(file_path, "rb") as file_obj:
        colect.add_file_to_blob_container(file_path, file_obj)

colect.create_user_search_index()
colect.create_data_source_connection()
colect.create_user_skillset()
colect.create_user_indexer()


2025-12-01 12:10:53.869 | INFO     | embeddings:get_container:66 - Container 'brussels-docs' already exists.
2025-12-01 12:10:54.002 | INFO     | embeddings:add_file_to_blob_container:96 - Uploaded 'F33073064.pdf' to container 'brussels-docs'.
2025-12-01 12:10:54.030 | INFO     | embeddings:add_file_to_blob_container:96 - Uploaded 'F33083732.pdf' to container 'brussels-docs'.
2025-12-01 12:10:54.047 | INFO     | embeddings:add_file_to_blob_container:96 - Uploaded 'F33091336.docx' to container 'brussels-docs'.
2025-12-01 12:10:54.065 | INFO     | embeddings:add_file_to_blob_container:96 - Uploaded 'F33094597.pdf' to container 'brussels-docs'.
2025-12-01 12:10:54.082 | INFO     | embeddings:add_file_to_blob_container:96 - Uploaded 'F33096176.pdf' to container 'brussels-docs'.
2025-12-01 12:10:54.101 | INFO     | embeddings:add_file_to_blob_container:96 - Uploaded 'F33097445.pdf' to container 'brussels-docs'.
2025-12-01 12:10:54.188 | INFO     | embeddings:add_file_to_blob_container:96 - 

In [22]:
colect.reset_and_run_indexer()

2025-12-01 12:12:33.812 | INFO     | embeddings:reset_and_run_indexer:252 - 🧹 Indexer 'brussels-docs-indexer' checkpoint reset successfully.
2025-12-01 12:12:34.013 | INFO     | embeddings:reset_and_run_indexer:256 - 🚀 Indexer 'brussels-docs-indexer' started successfully.


In [23]:
user_prompt = """
Extract the following fields ONLY from that document; do not invent facts.

Return ONE JSON object with exactly these keys and types:
- "title": string (Do not give back the document name. Instead, find what is the title that appears in the first page of document ?)
- "organisation": array of strings (What is the organizations that wrote this the document)
- "summary": string (summarise the document that you already possess in 4-6 sentences capturing the essentials; suitable for one spreadsheet cell, written in British English)
- "main themes": array of up to 10 short theme phrases (ranked by prominence, written in British English; do not pad with guesses)

Rules:
- Base your answer ONLY on the provided content; do not rely on outside knowledge.
- If a field is not present, set it to null (or [] for arrays).
- Preserve the document's original title (do not translate it); the "summary" and "main_themes" must be in British English.
- Output VALID JSON and NOTHING ELSE (no markdown fences, no prose, no citations like [doc1])."""


In [24]:
pillars_prompt = """Below are concise definitions of five challenges that motivate a new Biotechnology Act:

- VC (Access to Capital): Biotech requires large upfront funding for long R&D, trials, regulation, and scale-up; early bootstrapping helps, but VC/PE/strategic capital typically catalyze growth and Europe faces persistent risk-capital gaps and foreign exits.
- Clusters (Biotech clusters & technology centers): Co-located hubs concentrate talent, infrastructure, and investment, enabling public–private partnerships, sector specialization, shared services (regulatory/funding/market access), and internationalization.
- Skills (Reskilling & upskilling): Biomanufacturing needs deep process expertise (e.g., fermentation, molecular methods) plus cross-cutting data/AI, sustainability, systems thinking, and entrepreneurship; agile VET/HE and lifelong learning are critical.
- AI/data (Use of data & AI): AI (incl. GenAI) turns large scientific/process datasets into gains across discovery, engineering, and operations (e.g., target ID, CRISPR guide design, digital twins), with outcomes dependent on data quality, validation, and governance.
- Dual-use/security (Awareness of misuse risks): Attention to potential dual-use concerns and biosecurity, including risks of misuse such as development of bioweapons, uncontrolled pathogens, or other harmful applications. May appear in the context of regulation, funding decisions, or calls for responsible innovation and safeguards.


TASK: For each challenge, check whether the document makes a substantive reference to it. 
If yes, return a MAXIMUM TWO-SENTENCE summary in BRITISH ENGLISH of how the document addresses that challenge (what it says, proposes, or evidences). If not present, set the value to null.
Return ONE JSON object with EXACTLY these keys (values are either a string ≤2 sentences or null):
{
  "VC": <string or null>,
  "clusters": <string or null>,
  "skills": <string or null>,
  "AI/data": <string or null>,
  "dual_use/security": <string or null>
}

Rules:
- Base your answer ONLY on the provided content; do not rely on outside knowledge.
- Do not invent facts; if uncertain, use null.
- Output VALID JSON and NOTHING ELSE (no markdown fences, no prose, no citations)."""


In [25]:
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_COGNITIVE_SERVICES_ENDPOINT"],
    api_key=os.environ["AZURE_COGNITIVE_API"],
    api_version="2024-10-21",  # 2024-02-01+ supports data_sources
)

### FIRST TEST BY FILTERING FOR A SINGLE DOCUMENT

In [26]:

def odata_escape(s: str) -> str:
    # OData string literals escape single quotes by doubling them
    return s.replace("'", "''")

In [32]:
#doc_name = 'F3554271-Koppert submission for the biotech act consultations.pdf'        
doc_name="F33114469.pdf"

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "user", 
               "content": user_prompt}],
    extra_body={
        "data_sources": [
            {
                "type": "azure_search",
                "parameters": {
                    "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                    "index_name": f"{container_name}_index",
                    "authentication": {
                        "type": "api_key",
                        "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                    },

                    # Map fields so citations show nice titles/paths (optional but handy)
                    "fields_mapping": {
                        "title_field": "title",
                        "filepath_field": "filepath",
                        "url_field": "url",
                        
                    },

                    # B) If you stored the doc name in a custom field
                    "filter": f"title eq '{odata_escape(doc_name)}'",


                    "in_scope": True,        # keep the answer grounded to retrieved docs
                    # "strictness": 4,       # (optional) tighten relevance
                    # "top_n_documents": 5,  # (optional) fewer chunks for stricter focus
                },
            }
        ],
    },
)

print(completion.choices[0].message.content)


{
  "title": null,
  "organisation": [],
  "summary": null,
  "main themes": []
}


In [31]:
def extract_information(doc_name):
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": f"You already possess the document {doc_name} that contain all necessary information to answer following questions. \n" + user_prompt, }],
        response_format={"type": "json_object"},
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                        "index_name": f"{container_name}_index",
                        "authentication": {
                            "type": "api_key",
                            "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                        },
                        "fields_mapping": {
                            "title_field": "title",
                            "filepath_field": "filepath",
                            "url_field": "url",
                        },
                        "filter": f"title eq '{odata_escape(doc_name)}'",
                        "in_scope": True,
                        # "strictness": 4,
                        # "top_n_documents": 5,
                    },
                }
            ],
        },
    )
    raw = completion.choices[0].message.content or ""
    try:
        return json.loads(raw)  # now it should be valid JSON
    except json.JSONDecodeError:
        # Fallback: return your schema with nulls so the loop doesn't crash
        return {
            "title": None,
            "organization": [],
            "summary": None,
            "main themes": [],
        }


In [30]:
def extract_pillars(doc_name):
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": f"You already possess the document {doc_name} that contain all necessary information to answer following questions. \n" + pillars_prompt, }],
        response_format={"type": "json_object"},
        extra_body={
            "data_sources": [
                {
                    "type": "azure_search",
                    "parameters": {
                        "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                        "index_name": f"{container_name}_index",
                        "authentication": {
                            "type": "api_key",
                            "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                        },
                        "fields_mapping": {
                            "title_field": "title",
                            "filepath_field": "filepath",
                            "url_field": "url",
                        },
                        "filter": f"title eq '{odata_escape(doc_name)}'",
                        "in_scope": True,
                        # "strictness": 4,
                        # "top_n_documents": 5,
                    },
                }
            ],
        },
    )
    raw = completion.choices[0].message.content or ""
    try:
        return json.loads(raw)  # now it should be valid JSON
    except json.JSONDecodeError:
        # Fallback: return your schema with nulls so the loop doesn't crash
        return {
            "VC": None,
            "clusters": None,
            "skills": None,
            "AI/data": None,
            "dual_use/security": None
        }


In [34]:
doc_name='F33114405.pdf'
extract_information(doc_name)

{'title': 'VTT messages for the EU Biotechnology act',
 'organisation': ['VTT Technical Research Centre of Finland'],
 'summary': "The document presents VTT's recommendations for the development of the European Biotech Act, emphasising the need for efficient feedstock use, diversified production, and circular economy solutions to support economic security and ecosystem protection. It highlights the importance of aligning regulation, innovation, and industrial policies to fully harness biotechnology's potential for market products and European competitiveness. Key areas include investment in computational design, DNA engineering, bioprocess scale-up, and the integration of AI and machine learning. The document calls for improved access to capital, streamlined regulatory processes, and the creation of supportive infrastructure for biomanufacturing. It also details VTT's role in supporting start-ups and scale-ups through incubation, funding, and access to technology infrastructures. Examp

## LOOP OVER ALL DOCUMENTS

In [35]:

from time import sleep, strftime
from openai import APIConnectionError, BadRequestError
import re

def call_once_with_wait(fn, *args, delay=10, label=None, **kwargs):
    who = f"[{label}]" if label else ""
    print(f"{strftime('%H:%M:%S')} {who} waiting {delay}s before call...", flush=True)
    sleep(delay)
    try:
        out = fn(*args, **kwargs)
        print(f"{strftime('%H:%M:%S')} {who} call OK", flush=True)
        return out
    except BadRequestError as e:
        s = str(e)
        if "Rate limit is exceeded" in s:
            m = re.search(r"Try again in\s+(\d+)\s+seconds?", s, re.I)
            wait_s = int(m.group(1)) + 1 if m else max(5, delay)
            print(f"{strftime('%H:%M:%S')} {who} 429 rate limit → waiting {wait_s}s, then retrying once...", flush=True)
            sleep(wait_s)
            out = fn(*args, **kwargs)
            print(f"{strftime('%H:%M:%S')} {who} retry OK", flush=True)
            return out
        print(f"{strftime('%H:%M:%S')} {who} BadRequestError (not rate-limit): {e}", flush=True)
        raise
    except APIConnectionError as e:
        print(f"{strftime('%H:%M:%S')} {who} connection error → waiting 5s, then retrying once... ({e})", flush=True)
        sleep(5)
        out = fn(*args, **kwargs)
        print(f"{strftime('%H:%M:%S')} {who} retry OK", flush=True)
        return out
    except Exception as e:
        print(f"{strftime('%H:%M:%S')} {who} unexpected error: {e}", flush=True)
        raise


In [ ]:
dir_path = Path(data_path)  # folder_path already exists
docs = sorted([p.name for p in dir_path.iterdir() if p.is_file()])

results = []
for doc in docs:
    doc_name = doc
    info    = call_once_with_wait(extract_information, doc_name, delay=10, label=f"info:{doc_name}")
    pillars = call_once_with_wait(extract_pillars, doc_name, delay=10, label=f"pillars:{doc_name}")
    results.append({**info, **pillars})

In [42]:
from tqdm.auto import tqdm

dir_path = Path(data_path)
docs = sorted([p.name for p in dir_path.iterdir() if p.is_file()])

results, errors = [], []
pbar = tqdm(docs, desc="Processing documents", unit="doc", total=len(docs))

for doc in pbar:
    pbar.set_postfix_str(doc)  # show current file name on the bar
    try:
        info    = call_once_with_wait(extract_information, doc, delay=10, label=f"info:{doc}")
        pillars = call_once_with_wait(extract_pillars,     doc, delay=10, label=f"pillars:{doc}")
        results.append({**info, **pillars})
    except Exception as e:
        errors.append((doc, repr(e)))
        pbar.write(f"[ERROR] {doc}: {e}")

# Optional: inspect errors afterwards
# print(errors)


c:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\AIPC_general_app\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing documents:   0%|          | 0/74 [00:00<?, ?doc/s, F33073064.pdf]

13:44:17 [info:F33073064.pdf] waiting 10s before call...
13:44:34 [info:F33073064.pdf] call OK
13:44:34 [pillars:F33073064.pdf] waiting 10s before call...
13:44:50 [pillars:F33073064.pdf] call OK


Processing documents:   1%|▏         | 1/74 [00:32<39:57, 32.84s/doc, F33083732.pdf]

13:44:50 [info:F33083732.pdf] waiting 10s before call...
13:45:06 [info:F33083732.pdf] call OK
13:45:06 [pillars:F33083732.pdf] waiting 10s before call...
13:45:20 [pillars:F33083732.pdf] call OK


Processing documents:   3%|▎         | 2/74 [01:02<37:11, 30.99s/doc, F33091336.docx]

...

Processing documents:  97%|█████████▋| 72/74 [36:06<00:57, 28.90s/doc, F33117687.pdf]

14:20:23 [info:F33117687.pdf] waiting 10s before call...
14:20:39 [info:F33117687.pdf] call OK
14:20:39 [pillars:F33117687.pdf] waiting 10s before call...
14:20:52 [pillars:F33117687.pdf] call OK


Processing documents:  99%|█████████▊| 73/74 [36:34<00:28, 28.82s/doc, F3593805.pdf] 

14:20:52 [info:F3593805.pdf] waiting 10s before call...
14:21:08 [info:F3593805.pdf] call OK
14:21:08 [pillars:F3593805.pdf] waiting 10s before call...
14:21:21 [pillars:F3593805.pdf] call OK


Processing documents: 100%|██████████| 74/74 [37:03<00:00, 30.05s/doc, F3593805.pdf]


In [43]:
# --- make a DataFrame with your exact columns ---
def _tolist(x):
    if isinstance(x, list):
        return x
    return [x] if x else []

def _tostr_or_none(x):
    if x is None:
        return None
    return x if isinstance(x, str) else str(x)

rows = []
for r in results:
    rows.append({
        "title": r.get("title"),
        "organisation": _tolist(r.get("organisation")),     # keep as list
        "page_number": r.get("page_number"),
        "summary": r.get("summary"),
        "main themes": _tolist(r.get("main themes")),       # keep as list
        "VC": _tostr_or_none(r.get("VC")),                  # strings or None
        "clusters": _tostr_or_none(r.get("clusters")),
        "skills": _tostr_or_none(r.get("skills")),
        "AI/data": _tostr_or_none(r.get("AI/data")),
        "dual_use/security":_tostr_or_none(r.get("dual_use/security"))
    })

df = pd.DataFrame(
    rows,
    columns=["title", "organisation", "page_number", "summary", "main themes",
             "VC", "clusters", "skills", "AI/data","dual_use/security"]
)

In [44]:
df

,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
0,Government’s vision on Biotechnology 2025 - 2040,"[Ministry of Economic Affairs, Ministry of Inf...",None,This document outlines the Dutch government's ...,"[Long-term biotechnology strategy, Economic an...",The document highlights the persistent funding...,The document emphasises the importance of impr...,The document notes the need for sufficient tal...,The document discusses the increasing risks po...,The document addresses dual-use concerns by no...
1,Biotechnology for a more sustainable wood econ...,"[European Commission, Joint Research Centre, U...",None,This document discusses the role of biotechnol...,"[Sustainable wood economy, Wood preservation t...",The document highlights that long regulatory t...,None,None,None,None
2,RENEWABLE RECYCLED RESPONSIBLE EUROPEAN,[CEPI],None,The document is a contribution from CEPI urgin...,"[Broader scope for the European Biotech Act, F...",The document highlights that without targeted ...,The document advocates for the full integratio...,None,None,None
3,BIC contribution to the call for evidence on E...,[BIC],None,The document advocates for expanding the scope...,"[Expanding the EU Biotech Act scope, Biomanufa...",The document highlights the need for tailored ...,None,The document stresses the importance of invest...,None,None
4,TORINO STATEMENT,[],None,The document highlights the importance of the ...,"[circular bioeconomy, regulatory challenges, b...",The document highlights a persistent lack of f...,The document emphasises the importance of crea...,The document stresses the need for enough comp...,"The document identifies data as key, advocatin...",None
...,...,...,...,...,...,...,...,...,...,...
69,From Lab to Market: How the Biotech Act Can Ac...,[Fraunhofer Group for Resource Technologies an...,None,This document discusses the importance of the ...,"[Bridging research and industry, De-risking co...",The document highlights that financing for hig...,It proposes the establishment of Europe-wide j...,The document stresses the importance of empowe...,None,None
70,CO2 Value Europe’s response to the Commission’...,[CO2 Value Europe AISBL],None,This document is CO2 Value Europe’s response t...,"[Inclusion of CCU in the Biotech Act, Regulato...",The document highlights the need for clear fun...,None,None,None,None
71,SIOPE Response to EU Biotech Act Consultation ...,[SIOPE],None,The document compiles sources relevant to the ...,"[Paediatric oncology, CAR T-cell therapy, Biot...",None,None,None,None,None
72,TRACELESS MATERIALS | Position Paper. Biotech ...,[traceless],None,This position paper addresses regulatory chall...,"[Regulatory barriers for natural polymers, Pac...",The document highlights that the natural polym...,None,None,None,None


### Using PdfReader for page count

In [46]:
paths = sorted([p for p in Path(data_path).iterdir() if p.is_file()])[:len(df)]

# Optional: DOCX/DOC via Microsoft Word (Windows only)
try:
    WORD_AVAILABLE = True
    word = win32.gencache.EnsureDispatch("Word.Application")
    word.Visible = False
except Exception:
    WORD_AVAILABLE = False
    word = None

def count_pages(p: Path):
    suf = p.suffix.lower()
    if suf == ".pdf":
        try:
            with open(p, "rb") as f:
                r = PdfReader(f)
                if getattr(r, "is_encrypted", False):
                    try: r.decrypt("")
                    except Exception: return np.nan
                return len(r.pages)
        except Exception:
            return np.nan
    if suf in (".docx", ".doc") and WORD_AVAILABLE:
        try:
            doc = word.Documents.Open(str(p))
            pages = int(doc.ComputeStatistics(2))  # 2 = wdStatisticPages
            doc.Close(False)
            return pages
        except Exception:
            return np.nan
    return np.nan

page_counts = pd.Series([count_pages(p) for p in paths], index=df.index).astype("Int64")

if word: 
    try: word.Quit()
    except Exception: pass

if "page_number" in df.columns:
    df = df.drop(columns=["page_number"])
df.insert(2, "page_number", page_counts)


df.insert(0, "file_name", pd.Series(docs[:len(df)], index=df.index))


Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong poin

In [48]:
len(df)

74

In [49]:
# Rename column
df = df.rename(columns={"file_name": "feedback_reference"})

# Split at the period and take the first chunk
df["feedback_reference"] = df["feedback_reference"].str.split(".", n=1).str[0]
df.head()

,feedback_reference,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
0,F33073064,Government’s vision on Biotechnology 2025 - 2040,"[Ministry of Economic Affairs, Ministry of Inf...",51,This document outlines the Dutch government's ...,"[Long-term biotechnology strategy, Economic an...",The document highlights the persistent funding...,The document emphasises the importance of impr...,The document notes the need for sufficient tal...,The document discusses the increasing risks po...,The document addresses dual-use concerns by no...
1,F33083732,Biotechnology for a more sustainable wood econ...,"[European Commission, Joint Research Centre, U...",3,This document discusses the role of biotechnol...,"[Sustainable wood economy, Wood preservation t...",The document highlights that long regulatory t...,None,None,None,None
2,F33091336,RENEWABLE RECYCLED RESPONSIBLE EUROPEAN,[CEPI],2,The document is a contribution from CEPI urgin...,"[Broader scope for the European Biotech Act, F...",The document highlights that without targeted ...,The document advocates for the full integratio...,None,None,None
3,F33094597,BIC contribution to the call for evidence on E...,[BIC],2,The document advocates for expanding the scope...,"[Expanding the EU Biotech Act scope, Biomanufa...",The document highlights the need for tailored ...,None,The document stresses the importance of invest...,None,None
4,F33096176,TORINO STATEMENT,[],2,The document highlights the importance of the ...,"[circular bioeconomy, regulatory challenges, b...",The document highlights a persistent lack of f...,The document emphasises the importance of crea...,The document stresses the need for enough comp...,"The document identifies data as key, advocatin...",None


In [50]:
output_folder = Path(folder_path).parent / "results"
xlsx_path = Path(output_folder) / "RAG_task3_results.xlsx"

df.to_excel(xlsx_path, index=False)
print(f"Saved: {xlsx_path}")

Saved: C:\Users\YağmurAslan\OneDrive - Technopolis Group Ltd\RAG_Brussels\results\RAG_task3_results.xlsx


In [47]:
df[-30:]

,file_name,title,organisation,page_number,summary,main themes,VC,clusters,skills,AI/data,dual_use/security
44,F33114703.pdf,Strategic Vision Bioeconomy,[International Advisory Council on Global Bioe...,6,"This document, authored by the IACGB European ...","[EU bioeconomy strategy, Biotechnology innovat...",The document highlights persistent risk-capita...,It advocates for the creation and support of b...,The document stresses the importance of compre...,None,The document proposes establishing an EU Life ...
45,F33114714.pdf,Cefic position paper on biotechnology and biom...,[Cefic Bio-based Cluster],7,This document outlines Cefic's recommendations...,[Biotechnology and biomanufacturing definition...,The document highlights the persistent risk of...,It calls for the enhancement of the recently c...,The document proposes developing EU-wide biote...,It advocates for a 'data initiative' to provid...,The document notes the relevance of biomanufac...
46,F33114715.pdf,Additional information,"[Dyrenes Beskyttelse, Animal Protection Denmark]",2,The document discusses the potential of biosol...,"[Biosolutions and sustainability, Cellular agr...",None,None,None,None,None
47,F33114716.pdf,Catalonia.Health Position on the EU Biotech Ac...,[Catalonia.Health],8,This document presents Catalonia.Health's posi...,"[EU Biotech Act advocacy, Decline in European ...",The document highlights that European biotech ...,Catalonia.Health is presented as a successful ...,The document calls for EU-wide training progra...,The document recommends establishing EU clinic...,None
48,F33114718.pdf,Position on Biosecurity Regulation for DNA Dat...,"[DNA Data Storage Alliance (DDSA), SNIA]",5,This document presents the DNA Data Storage Al...,"[Biosecurity regulation for DNA data storage, ...",None,None,None,None,The document discusses biosecurity concerns sp...
49,F33114871.pdf,Gerichtstraße 49 13347 Berlin Germany,"[COGEM, ZKBS]",50,This document is a joint open letter from the ...,[Environmental risk assessment of samRNA and V...,None,None,None,None,The document discusses the challenges posed by...
50,F33114902.pdf,Unlocking Europe’s Bioeconomy: Why Policy supp...,"[Kemira, IFF]",3,The document discusses the importance of suppo...,"[EU bioeconomy policy, Biobased polymers and p...",None,None,None,None,None
51,F33114919.docx,HOUSE OF THE | DUTCH,"[Ministerium für Umwelt, Klima und Energiewirt...",3,The document presents a position paper from th...,"[Biotechnology initiative, Sustainable circula...",None,The document highlights the importance of a tr...,None,None,None
52,F33114933.pdf,Finnish Food and Drink Industries’ input for p...,[Finnish Food and Drink Industries’ Federation],2,The document provides input from the Finnish F...,"[Biotechnology in food industry, Regulatory en...",The document highlights the need for RDI fundi...,None,None,None,None
53,F33114947.pdf,Réponse à la consultation publique sur le règl...,[],2,The document raises critical questions about t...,"[Societal challenges and solutions, Biotechnol...",None,None,None,None,The document highlights the necessity of stren...


### Treating the Missings

In [36]:
output_folder = Path(folder_path).parent / "results"
xlsx_path = Path(output_folder) / "RAG_all_results.xlsx"
df = pd.read_excel(xlsx_path)

In [38]:
       
doc_name='F3565290-UniSR_European Biotech Act_Extended.pdf'
#doc_name='F3564682-Reco EU biotech EWPA EWPM 10.06.2025.pdf'

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "user", 
               "content": user_prompt}],
    extra_body={
        "data_sources": [
            {
                "type": "azure_search",
                "parameters": {
                    "endpoint": os.environ["AZURE_AI_SEARCH_ENDPOINT"],
                    "index_name": f"{container_name}_index",
                    "authentication": {
                        "type": "api_key",
                        "key": os.environ["AZURE_AI_SEARCH_API_KEY"],
                    },

                    # Map fields so citations show nice titles/paths (optional but handy)
                    "fields_mapping": {
                        "title_field": "title",
                        "filepath_field": "filepath",
                        "url_field": "url",
                        
                    },

                    # B) If you stored the doc name in a custom field
                    "filter": f"title eq '{odata_escape(doc_name)}'",


                    "in_scope": True,        # keep the answer grounded to retrieved docs
                    # "strictness": 4,       # (optional) tighten relevance
                    # "top_n_documents": 5,  # (optional) fewer chunks for stricter focus
                },
            }
        ],
    },
)

print(completion.choices[0].message.content)

{
  "title": "Contribution to the Consultation on the Future European Biotech Act",
  "organisation": ["Università Vita-Salute San Raffaele"],
  "summary": "The document is a contribution from Università Vita-Salute San Raffaele to the European Commission's consultation on the future European Biotech Act. It welcomes the initiative and provides recommendations to strengthen Europe's position in biotechnology. Key suggestions include enhanced support for academic centres, streamlined regulatory processes, and the creation of a 'Single Biotech Permit' for EU-wide operations. The document also advocates for a dedicated translational track for biotech projects and harmonised tax incentives for private investment. Additionally, it highlights the importance of secure and efficient access to anonymised patient data. The overall aim is to reduce barriers and accelerate innovation in the European biotech sector.",
  "main themes": [
    "Support for academic biotech research",
    "Regulatory s

In [39]:

extract_pillars(doc_name)

{'VC': 'The document proposes an EU framework for tax relief to incentivise high-risk investment in biotech, including tax credits, capital gains tax deferrals, and favourable treatment of convertible instruments. It argues that harmonising such incentives would create a more predictable and attractive landscape for both European and international investors.',
 'clusters': 'The document recommends dedicated support for academic centres to play a leading role in future biotech clusters and centres of excellence across the EU. It also suggests the creation of themed pan-European biotech accelerators to systematically transform research outputs into companies with access to capital, talent, and infrastructure.',
 'skills': None,
 'AI/data': 'The document highlights the importance of real-world data and AI in accelerating the development of innovative therapies, calling for dedicated support for AI technologies that leverage such data. It emphasises the need for collaboration between biote

In [40]:
main_themes = [
    "Support for academic biotech research",
    "Regulatory streamlining",
    "Single Biotech Permit",
    "Translational track for biotech projects",
    "Tax incentives for biotech investment",
    "Data access and security",
    "Innovation acceleration",
    "EU-wide harmonisation"
  ]

print(main_themes)

['Support for academic biotech research', 'Regulatory streamlining', 'Single Biotech Permit', 'Translational track for biotech projects', 'Tax incentives for biotech investment', 'Data access and security', 'Innovation acceleration', 'EU-wide harmonisation']
